In [2]:
import pdfplumber
import pymupdf
import pandas as pd

In [3]:


# Path to the PDF file
pdf_path = r'E:\RAG_Project\data\raw\0001558370-19-000470.pdf'

# Extracting text using PyMuPDF
def extract_text_pymupdf(pdf_path):
    document = pymupdf.open(pdf_path)
    text = ""
    for page_num in range(len(document)):
        page = document.load_page(page_num)
        text += page.get_text()
    return text

# Extracting tables using pdfplumber
def extract_tables_pdfplumber(pdf_path):
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables_on_page = page.extract_tables()
            tables.extend(tables_on_page)
    return tables

# Extract text and tables
text = extract_text_pymupdf(pdf_path)
tables = extract_tables_pdfplumber(pdf_path)




Extracted Text:
low
 
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
 
FORM 10-K
 
☒   ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE
SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2018
 
Commission file number 1-3285
 
3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employer Identification No. 41-0417775
Principal executive offices: 3M Center, St. Paul, Minnesota 55144
Telephone number: (651) 733-1110
 
SECURITIES REGISTERED PURSUANT TO SECTION 12(b) OF THE ACT:
 
 
 
Title of each class
 
Name of each exchange
on which registered
Common Stock, Par Value $.01 Per Share
 
1.500% Notes due 2026
Floating Rate Notes due 2020
0.375% Notes due 2022
0.950% Notes due 2023
1.750% Notes due 2030
1.500% Notes due 2031
 
New York Stock Exchange, Inc.
Chicago Stock Exchange, Inc.
New York Stock Exchange, Inc.
New York Stock Exchange, Inc.
New York Stock Exchange, Inc.
New York Stock Exchange, Inc.
New York Stock Exchange, Inc.
New York St

In [4]:
# Display extracted text
print("Extracted Text:")
print(text[:20])  # Display the first 2000 characters

# Display extracted tables
print("\nExtracted Tables:")
for table in tables:
    df = pd.DataFrame(table[1:], columns=table[0])  # Assuming first row is header
    print(df)

Extracted Text:
low
 
UNITED STATES


Extracted Tables:
                     Name         Age        \
0         Inge. G. Thulin          65         
1                          None  None  None   
2        Michael F. Roman          59         
3                          None  None  None   
4        John P. Banovetz          51         
5                          None  None  None   
6         James L. Bauman          59         
7                          None  None  None   
8        Julie L. Bushman          57         
9                          None  None  None   
10        Joaquin Delgado          58         
11                         None  None  None   
12           Ivan K. Fong          57         
13                         None  None  None   
14  Nicholas C. Gangestad          54         
15                         None  None  None   
16         Eric D. Hammes          44         
17                         None  None  None   
18           Paul A. Keel          49         
19  

In [11]:
tables[30]

[['Full Year 2017 GAAP',
  '',
  '$',
  '31,657',
  '',
  '$',
  '7,692',
  '',
  '24.3',
  '%',
  '$',
  '7,548',
  '',
  '$',
  '2,679',
  '',
  '35.5',
  '%',
  '$',
  '4,858',
  '',
  '$',
  '7.93',
  '',
  '',
  '']]

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd

# URL to the HTML file
html_url = 'https://www.sec.gov/Archives/edgar/data/66740/000155837019000470/mmm-20181231x10k.htm'

# Path to the ChromeDriver executable
chrome_driver_path =r'E:\RAG_Project\utils\chrome_driver\chromedriver-win32\chromedriver.exe'

# Set up Selenium WebDriver
options = Options()
options.headless = True  # Run in headless mode
service = Service(chrome_driver_path)
driver = webdriver.Chrome(service=service, options=options)

# Function to fetch HTML content using Selenium
def fetch_html_content(html_url):
    driver.get(html_url)
    return driver.page_source

# Fetch HTML content from the specified URL
html_content = fetch_html_content(html_url)

# Parse the HTML content using BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

# Extract all text
text = soup.get_text(separator='\n')

# Extract all tables
tables = []
for table in soup.find_all('table'):
    rows = []
    for row in table.find_all('tr'):
        cells = row.find_all(['td', 'th'])
        cells_text = [cell.get_text(strip=True) for cell in cells]
        rows.append(cells_text)
    tables.append(pd.DataFrame(rows))

# Close the Selenium WebDriver
driver.quit()

# Display extracted text
print("Extracted Text:")
print(text[:2000])  # Display the first 2000 characters

# Display extracted tables
print("\nExtracted Tables:")
for df in tables:
    print(df)


Extracted Text:


10-K

1

mmm-20181231x10k.htm

10-K








			mmm_Current_Folio_10K
		








low






 






UNITED STATES






SECURITIES AND EXCHANGE COMMISSION






Washington, D.C. 20549






 






FORM 10-K






 






☒   ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE






SECURITIES EXCHANGE ACT OF 1934






For the fiscal year ended December 31, 2018






 






Commission file number 1-3285






 






3M COMPANY












 








 








 












State of Incorporation: 
Delaware








 








I.R.S. Employer Identification No. 
41-0417775










Principal executive offices: 
3M Center, St. Paul, Minnesota 55144






Telephone number: 
(651) 733-1110






 






SECURITIES REGISTERED PURSUANT TO SECTION 12(b) OF THE ACT:












 








 








 












Title of each class








 








Name of each exchange
on which registered












Common Stock, Par Value $.01 Per Share




 




1.500% Notes due 2026





In [28]:
# Function to clean extracted tables
def clean_tables(tables):
    cleaned_tables = []
    for table in tables:
        if len(table) > 1:  # Ensure there are rows to process
            df = pd.DataFrame(table[1:], columns=table[0])
            # Perform cleaning operations, e.g., removing empty rows, handling NaNs, etc.
            df.dropna(how='all', inplace=True)
            cleaned_tables.append(df)
    return cleaned_tables

# Clean the extracted tables
cleaned_tables = clean_tables(tables)

# Display cleaned tables
print("\nCleaned Tables:")
for df in cleaned_tables:
    print(df)
    print("\n")

# Optionally, display extracted text (commented out for clarity)
# print("Extracted Text:")
# print(text[:2000])  # Display the first 2000 characters


Cleaned Tables:
Empty DataFrame
Columns: [, State of Incorporation:Delaware]
Index: []


Empty DataFrame
Columns: [, Title of each class, Common Stock, Par Value $.01 Per Share1.500% Notes due 2026Floating Rate Notes due 20200.375% Notes due 20220.950% Notes due 20231.750% Notes due 20301.500% Notes due 2031]
Index: []


Empty DataFrame
Columns: [, Large accelerated filer  ☒, ]
Index: []


Empty DataFrame
Columns: [, , PART I, ITEM 1, , ITEM 1A, , ITEM 1B, , ITEM 2, , ITEM 3, , ITEM 4, , PART II, ITEM 5, , ITEM 6, , ITEM 7, , , , , , , , , , , , , ITEM 7A, , ITEM 8, , , , , , , , , , ]
Index: []

[0 rows x 46 columns]


Empty DataFrame
Columns: [, ITEM 8, , , , , , , , , , , , , , , , , , , , , , , , , , , , ITEM 9, , ITEM 9A, , ITEM 9B, , PART III, ITEM 10, , ITEM 11, , ITEM 12, , ITEM 13, , ITEM 14, , PART IV, ITEM 15, , ITEM 16, ]
Index: []

[0 rows x 51 columns]


Empty DataFrame
Columns: [, Name, Inge. G. Thulin, , Michael F. Roman, , John P. Banovetz, , James L. Bauman, , Julie 

In [34]:
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer

# Example of vectorizing text data using TF-IDF
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform([text])

# Initialize faiss index
index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors.toarray())

# Now you can perform similarity search on this index


In [ ]:
import os
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAI
from langchain.docstore import InMemoryDocstore
from langchain.docstore.document import Document
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Load OpenAI key from .env
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise EnvironmentError("OPENAI_API_KEY not set — add it to your .env file.")

documents = [Document(page_content=text)]
docstore = InMemoryDocstore({str(i): doc for i, doc in enumerate(documents)})
index_to_docstore_id = {i: str(i) for i in range(len(documents))}

vector_store = FAISS(index, docstore, index_to_docstore_id)

template = PromptTemplate(
    input_variables=["topic"],
    template="Extract information about {topic} from the given text.",
)
llm = OpenAI(api_key=openai_api_key)
llm_chain = LLMChain(llm=llm, prompt=template)

response = llm_chain.run({"topic": "net PP&E for FY2018"})
print(response)